# Train a tuned lens from an extraction

A tuned lens (Belrose et al., 2023) replaces the logit lens's identity map with one learned
affine translator per layer, trained so that each layer's decoded logits match the model's
final logits. This notebook trains one from a disk-mode `tl.extract_dataset` artifact, so the
training data carries its provenance: the stimulus ids and the model identity recorded in the
manifest. The trained translators then go to `torchlens.semantic.logit_lens(trace, lens=...)`,
which checks that the lens was built for the model it is applied to.

The model is a tiny randomly initialized GPT-shaped stack so the notebook runs offline in
seconds; with a real model, the same cells apply to its block addresses.

In [ ]:
# recipe-test: tool=torchlens expected=True
import torch
from torch import nn

import torchlens as tl
from torchlens.dataset_extraction import load_extraction
from torchlens.semantic import logit_lens


class Block(nn.Module):
    def __init__(self, d: int) -> None:
        super().__init__()
        self.attn, self.mlp = nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class Stack(nn.Module):
    def __init__(self, d: int, n: int) -> None:
        super().__init__()
        self.h = nn.ModuleList([Block(d) for _ in range(n)])
        self.ln_f = nn.LayerNorm(d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.h:
            x = block(x)
        return self.ln_f(x)


class TinyLM(nn.Module):
    def __init__(self, d: int = 8, n: int = 3, vocab: int = 11) -> None:
        super().__init__()
        self.transformer = Stack(d, n)
        self.lm_head = nn.Linear(d, vocab, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lm_head(self.transformer(x))


torch.manual_seed(0)
model = TinyLM().eval()
stimuli = torch.randn(96, 4, 8)  # (stimulus, position, embedding)
RESULT = model(stimuli).shape == (96, 4, 11)

## 1. Extract the training data with provenance

Every block output plus the final logits, one artifact, with stimulus ids.

In [ ]:
# recipe-test: tool=torchlens expected=True
blocks = [f"transformer.h.{i}" for i in range(3)]
tl.extract_dataset(
    model,
    stimuli,
    [*blocks, "lm_head"],
    batch_size=32,
    path="lens_data",
    stimulus_ids=[f"seq{i:03d}" for i in range(len(stimuli))],
    progress=False,
)
data = load_extraction("lens_data")
identity = data.manifest["signature"]["model_identity"]  # the measured model digest
RESULT = set(data.activations) == {*blocks, "lm_head"} and identity["level"] == "measured"

## 2. Train one affine translator per layer

Each translator maps a block's hidden state to the final hidden space, then decodes through
the model's own final norm and unembedding. The loss is the KL divergence from the model's
final next-token distribution. The first 64 stimuli train, the last 32 evaluate.

In [ ]:
# recipe-test: tool=torchlens expected=True
class Translator(nn.Module):
    """hidden -> logits through a learned affine map and the model's own head."""

    def __init__(self, d: int, model_identity: dict) -> None:
        super().__init__()
        self.affine = nn.Linear(d, d)
        nn.init.eye_(self.affine.weight)
        nn.init.zeros_(self.affine.bias)
        self.model_identity = model_identity  # logit_lens checks this against the traced model

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        return model.lm_head(model.transformer.ln_f(self.affine(hidden)))


def kl_to_final(logits: torch.Tensor, final: torch.Tensor) -> torch.Tensor:
    return nn.functional.kl_div(
        logits.log_softmax(-1), final.log_softmax(-1), log_target=True, reduction="batchmean"
    )


final = data.activations["lm_head"]
train, held_out = slice(0, 64), slice(64, 96)
for p in model.parameters():
    p.requires_grad_(False)

lenses, tuned_kl, plain_kl = {}, {}, {}
for address in blocks:
    hidden = data.activations[address]
    translator = Translator(8, identity)
    optimizer = torch.optim.Adam(translator.parameters(), lr=0.02)
    for _ in range(200):
        optimizer.zero_grad()
        kl_to_final(translator(hidden[train]), final[train]).backward()
        optimizer.step()
    with torch.no_grad():
        tuned_kl[address] = kl_to_final(translator(hidden[held_out]), final[held_out]).item()
        plain = model.lm_head(model.transformer.ln_f(hidden[held_out]))
        plain_kl[address] = kl_to_final(plain, final[held_out]).item()
    lenses[address] = translator.eval()

RESULT = tuned_kl[blocks[0]] < plain_kl[blocks[0]]  # the tuned lens beats the logit lens early

## 3. Use the tuned lens on a new trace

`logit_lens` applies one translator per block. Because each translator carries the
manifest's measured model identity, `logit_lens` compares it with the traced model and would
refuse `logit_lens_identity_mismatch` for a different model.

In [ ]:
# recipe-test: tool=torchlens expected=('user', 3)
trace = tl.trace(model, stimuli[held_out], capture=tl.options.CaptureOptions(layers_to_save="all"))
result = logit_lens(trace, lens=lenses)
RESULT = (result.lens_source, len(result.entries))